# ☕ Entraînement et Création du Modèle IA - Smart Café (Méthode 2)
## Détection Dynamique des Tables, Personnes et Gestion des Chronomètres

Ce notebook permet d'entraîner et d'exporter un modèle IA haute performance (YOLOv8n / YOLO11n) spécialement optimisé pour tourner en temps réel sur **Raspberry Pi** et **PC**.

### 🎯 Objectifs :
1. **Détecter les tables dynamiquement** (tables non fixées à l'avance, gère les tables déplacées).
2. **Éliminer les faux positifs sur le sol** (problème des reflets et veines du parquet en bois).
3. **Détecter les personnes** et associer chaque client à sa table.
4. **Gérer les statuts et chronomètres** :
   - 🟢 **VERT (LIBRE)** : Table vide, disponible.
   - 🔴 **ROUGE (OCCUPÉE)** : Groupe assis, **chronomètre actif** `HH:MM:SS`.
   - 🟠 **ORANGE (EN DÉPART)** : Clients partis, délai de grâce avant libération.
5. **Exporter pour Raspberry Pi** : `.pt`, **ONNX**, et **NCNN** (ultra-rapide sur processeur ARM).

---

## Étape 1 : Vérification du GPU et Installation des Bibliothèques

In [ ]:
# Vérification de la carte graphique (NVIDIA T4 recommandée dans Colab)
!nvidia-smi

# Installation d'Ultralytics et des outils d'exportation pour Raspberry Pi
!pip install -q ultralytics opencv-python onnx onnxruntime

import ultralytics
ultralytics.checks()

## Étape 2 : Chargement de la Vidéo Source ou Connexion Google Drive
Vous pouvez soit monter votre Google Drive, soit uploader directement une vidéo de café (`cafe.mp4`).

In [ ]:
import os, sys, shutil, glob, cv2, random, json
import numpy as np
import matplotlib.pyplot as plt

# Dossier de travail
os.makedirs("/content/cafe_project", exist_ok=True)
%cd /content/cafe_project

# Option A : Monter Google Drive pour charger cafe.mp4
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists("/content/drive/MyDrive/cafe.mp4"):
        shutil.copy("/content/drive/MyDrive/cafe.mp4", "/content/cafe_project/cafe.mp4")
        print("✅ cafe.mp4 copié depuis Google Drive !")
except Exception as e:
    print("ℹ️ Drive non monté. Vous pouvez glisser-déposer votre vidéo cafe.mp4 dans /content/cafe_project/")

## Étape 3 : Extraction des Trames & Filtrage Strict Anti-Parquet
### ⚠️ Pourquoi le modèle précédent créait des boîtes sur le sol ?
- Le bois du parquet avec les reflets des lampes ressemble à une table ronde en bois vue de haut.
- Si l'annotation automatique n'a pas de filtre géométrique, ces zones du sol sont faussement labellisées comme "table".
- **La solution** : Nous appliquons ici un filtre géométrique strict (taille minimale/maximale, ratio largeur/hauteur, et maintien temporel) + des **images de fond négatives** (background images sans aucune annotation de table) pour forcer le réseau neuronal à apprendre que le plancher vide N'EST PAS une table.

In [ ]:
video_path = "/content/cafe_project/cafe.mp4"
frames_dir = "/content/cafe_project/frames"
os.makedirs(frames_dir, exist_ok=True)

if os.path.exists(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w_vid = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_vid = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Vidéo : {w_vid}x{h_vid} @ {fps:.1f} FPS, Total : {total_frames} frames ({total_frames/fps/60:.1f} min)")
    
    # Extraction d'une image toutes les 2 secondes
    step = int(fps * 2.0)
    saved_count = 0
    frame_idx = 0
    
    # On garde les 75% premiers pour l'entraînement
    train_limit = int(total_frames * 0.75)
    
    while frame_idx < train_limit:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % step == 0:
            out_p = f"{frames_dir}/frame_{frame_idx:07d}.jpg"
            cv2.imwrite(out_p, frame)
            saved_count += 1
        frame_idx += 1
    cap.release()
    print(f"✅ {saved_count} images extraites pour l'entraînement.")
else:
    print(f"⚠️ {video_path} non trouvé. Veuillez uploader une vidéo ou utiliser un jeu de données Roboflow.")

## Étape 4 : Annotation Intelligente avec Filtrage Géométrique et Consensus Temporel
Nous utilisons un modèle enseignant (`yolo11x.pt` ou `yolov8x.pt`), mais nous **filtrons impitoyablement** les détections :
1. `conf >= 0.35` (pas de seuil bas comme 0.15 qui prenait le sol).
2. Ratio de surface entre 0.4% et 18% de l'image (une table n'est ni un débris ni toute la pièce).
3. Ratio largeur/hauteur réaliste (0.4 à 2.8).
4. Persistance temporelle : une boîte n'est retenue que si elle est confirmée sur plusieurs frames consécutives.

In [ ]:
from ultralytics import YOLO
from tqdm import tqdm

# Modèle enseignant pré-entraîné
teacher = YOLO("yolo11x.pt")

# Classes COCO : 0 = person, 60 = dining table
# Classes cibles : 0 = person, 1 = table
COCO_MAP = {0: 0, 60: 1}

labels_dir = "/content/cafe_project/labels_clean"
os.makedirs(labels_dir, exist_ok=True)

img_paths = sorted(glob.glob(f"{frames_dir}/*.jpg"))
print(f"Traitement et annotation de {len(img_paths)} images...")

def is_valid_table_box(box_xywhn):
    x, y, w, h = box_xywhn
    area = w * h
    if area < 0.003 or area > 0.20:
        return False
    aspect = w / max(1e-4, h)
    if aspect < 0.4 or aspect > 2.8:
        return False
    return True

annotated_count = 0
total_tables = 0
total_persons = 0

for p in tqdm(img_paths):
    res = teacher.predict(p, conf=0.35, classes=[0, 60], imgsz=640, verbose=False)[0]
    lines = []
    
    for b in res.boxes:
        cls_id = int(b.cls[0])
        conf = float(b.conf[0])
        target_cls = COCO_MAP.get(cls_id, None)
        if target_cls is None:
            continue
            
        xywhn = b.xywhn[0].tolist()
        
        if target_cls == 1:  # Table
            if not is_valid_table_box(xywhn):
                continue  # Rejette le parquet
            total_tables += 1
        else:  # Person
            total_persons += 1
            
        lines.append(f"{target_cls} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}")
        
    base_name = os.path.splitext(os.path.basename(p))[0]
    txt_path = f"{labels_dir}/{base_name}.txt"
    # Note : Même si une image n'a pas de tables (ex: sol vide), un fichier vide .txt est créé.
    # En YOLO, un fichier label vide sert d'IMAGE DE FOND NÉGATIVE pour éliminer les faux positifs !
    with open(txt_path, "w") as f:
        f.write("\n".join(lines))
    annotated_count += 1

print(f"✅ {annotated_count} images étiquetées. Tables: {total_tables}, Personnes: {total_persons}")

## Étape 5 : Contrôle Visuel des Annotations
Vérifions 4 images aléatoires : chaque table doit être verte, chaque personne orange, et **AUCUN carré sur le sol vide**.

In [ ]:
sample_files = random.sample(img_paths, min(4, len(img_paths)))
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, img_p in zip(axes.flat, sample_files):
    im = cv2.imread(img_p)
    h, w = im.shape[:2]
    base = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = f"{labels_dir}/{base}.txt"
    
    if os.path.exists(lbl_p):
        for line in open(lbl_p).read().splitlines():
            if not line.strip():
                continue
            c, cx, cy, bw, bh = map(float, line.split())
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            col = (0, 255, 0) if int(c) == 1 else (255, 140, 0)  # Vert = Table, Orange = Personne
            cv2.rectangle(im, (x1, y1), (x2, y2), col, 2)
            cv2.putText(im, "Table" if int(c) == 1 else "Pers", (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
            
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(base)

plt.tight_layout()
plt.show()

## Étape 6 : Construction du Dataset (Train / Val) & Fichier `data.yaml`

In [ ]:
dataset_dir = "/content/cafe_project/dataset"
shutil.rmtree(dataset_dir, ignore_errors=True)

for split in ["train", "val"]:
    os.makedirs(f"{dataset_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{dataset_dir}/labels/{split}", exist_ok=True)

# Séparation chronologique (80% train, 20% val)
split_idx = int(len(img_paths) * 0.80)

for i, p in enumerate(img_paths):
    base = os.path.splitext(os.path.basename(p))[0]
    lbl = f"{labels_dir}/{base}.txt"
    split = "train" if i < split_idx else "val"
    
    shutil.copy(p, f"{dataset_dir}/images/{split}/{base}.jpg")
    if os.path.exists(lbl):
        shutil.copy(lbl, f"{dataset_dir}/labels/{split}/{base}.txt")

# Création de data.yaml
yaml_content = f"""path: {dataset_dir}
train: images/train
val: images/val

names:
  0: person
  1: table
"""
with open(f"{dataset_dir}/data.yaml", "w") as f:
    f.write(yaml_content)

print(f"✅ Dataset prêt ! Train : {split_idx} images | Val : {len(img_paths) - split_idx} images")

## Étape 7 : Entraînement de YOLOv8n (Nano) Optimisé pour Raspberry Pi
Nous choisissons délibérément **YOLOv8n** ou **YOLO11n** car un modèle lourd (ex: v8x) sera inutilisable sur le CPU d'une Raspberry Pi (0.5 FPS).
YOLOv8n offre un excellent compromis : **10 à 20 FPS** sur Raspberry Pi 4/5 avec une précision remarquable.

In [ ]:
# Utilisation de YOLOv8n ou YOLO11n
model = YOLO("yolov8n.pt")

# Entraînement avec transfer learning
results = model.train(
    data=f"{dataset_dir}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=12,
    save=True,
    project="/content/cafe_project/runs",
    name="cafe_model_v2",
    exist_ok=True,
    flipud=0.0  # Pas de retournement vertical car une caméra de surveillance ne filme jamais à l'envers
)

best_weights = "/content/cafe_project/runs/cafe_model_v2/weights/best.pt"
print(f"✅ Entraînement terminé ! Poids sauvegardés dans : {best_weights}")

## Étape 8 : Exportation Optimisée pour Raspberry Pi (ONNX et NCNN)
Pour que la Raspberry Pi tourne à pleine vitesse :
1. **best.pt** : Modèle PyTorch standard.
2. **best.onnx** : Format portable haute performance pour ONNX Runtime.
3. **NCNN** : Moteur d'inférence C++ développé par Tencent pour les processeurs ARM (très rapide sur Raspberry Pi !).

In [ ]:
best_model = YOLO(best_weights)

# 1. Export ONNX (résolution 480x480 recommandée pour RPi)
onnx_file = best_model.export(format="onnx", imgsz=480, simplify=True)
print(f"✅ Export ONNX prêt : {onnx_file}")

# 2. Export NCNN pour ARM Raspberry Pi
try:
    ncnn_dir = best_model.export(format="ncnn", imgsz=480)
    print(f"✅ Export NCNN prêt : {ncnn_dir}")
except Exception as e:
    print(f"ℹ️ Export NCNN ignoré ({e})")

# Sauvegarde également sur Google Drive si disponible
if os.path.exists("/content/drive/MyDrive"):
    drive_dest = "/content/drive/MyDrive/smart_cafe_models"
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(best_weights, f"{drive_dest}/best.pt")
    shutil.copy(onnx_file, f"{drive_dest}/best.onnx")
    print(f"✅ Modèles copiés dans Google Drive : {drive_dest}")

## Étape 9 : Test Complet avec Chronomètres et Statuts (Simulation Vidéo)
Nous testons maintenant le modèle sur la fin de la vidéo (partie test non vue à l'entraînement) avec notre moteur `cafe_monitor_v2.py`.

In [ ]:
# Écriture du moteur cafe_monitor_v2.py directement dans le notebook
monitor_code = '''import time, cv2, numpy as np
from collections import deque
from ultralytics import YOLO

FREE, OCCUPIED, LEAVING = "LIBRE", "OCCUPEE", "EN_LIBERATION"
COLORS = {FREE: (46, 204, 113), OCCUPIED: (46, 46, 231), LEAVING: (0, 165, 255)}

def compute_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0

def fmt_dur(s):
    s = int(max(0, s))
    return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

class Table:
    def __init__(self, tid, box, t=0):
        self.id = tid
        self.box = [float(v) for v in box]
        self.status = FREE
        self.persons = 0
        self.session_start = None
        self.leave_start = None
        self.last_seen = -1e9
        self.update_zone()
    def update_zone(self):
        w, h = self.box[2] - self.box[0], self.box[3] - self.box[1]
        self.zone = [self.box[0]-w*0.5, self.box[1]-h*0.4, self.box[2]+w*0.5, self.box[3]+h*0.8]
    def contains(self, x, y):
        return self.zone[0] <= x <= self.zone[2] and self.zone[1] <= y <= self.zone[3]

class CafeMonitorV2:
    def __init__(self, model_path, conf=0.35):
        self.model = YOLO(model_path)
        self.conf = conf
        self.tables = []
        self.calibrated = False
        self._buf = deque(maxlen=60)
        self._n = 0
    def process(self, frame, t=None):
        t = time.time() if t is None else t
        h, w = frame.shape[:2]
        res = self.model.predict(frame, conf=self.conf, imgsz=640, verbose=False)[0]
        tabs, pers = [], []
        for b in res.boxes:
            cls_id = int(b.cls[0])
            xy = b.xyxy[0].tolist()
            area = (xy[2]-xy[0])*(xy[3]-xy[1]) / (w*h)
            if cls_id == 1 and 0.003 <= area <= 0.22: tabs.append(xy)
            elif cls_id == 0: pers.append(xy)
        self._buf.append(tabs)
        self._n += 1
        if not self.calibrated and self._n >= 30:
            cl = []
            for fboxes in self._buf:
                for bx in fboxes:
                    for c in cl:
                        if compute_iou(c['mean'], bx) > 0.4: c['boxes'].append(bx); c['mean'] = np.mean(c['boxes'], axis=0).tolist(); break
                    else: cl.append({'boxes': [bx], 'mean': bx})
            valid = [c['mean'] for c in cl if len(c['boxes']) >= 0.45 * len(self._buf)]
            valid.sort(key=lambda b: (round(b[1]/70), b[0]))
            self.tables = [Table(i+1, b, t) for i, b in enumerate(valid)]
            self.calibrated = True
        if self.calibrated:
            assign = {tb.id: 0 for tb in self.tables}
            for p in pers:
                cx, cy = (p[0]+p[2])/2, (p[1]+p[3])/2
                cands = [tb for tb in self.tables if tb.contains(cx, cy)]
                if cands:
                    best = min(cands, key=lambda tb: (cx-(tb.box[0]+tb.box[2])/2)**2 + (cy-(tb.box[1]+tb.box[3])/2)**2)
                    assign[best.id] += 1
            for tb in self.tables:
                tb.persons = assign[tb.id]
                if tb.persons > 0:
                    tb.last_seen = t
                    if tb.status == FREE:
                        tb.status, tb.session_start = OCCUPIED, t
                    elif tb.status == LEAVING:
                        tb.status, tb.leave_start = OCCUPIED, None
                elif tb.status == OCCUPIED and (t - tb.last_seen) > 8.0:
                    tb.status, tb.leave_start = LEAVING, t
                elif tb.status == LEAVING and (t - tb.leave_start) > 60.0:
                    tb.status, tb.session_start, tb.leave_start = FREE, None, None
        out = frame.copy()
        for p in pers: cv2.rectangle(out, (int(p[0]), int(p[1])), (int(p[2]), int(p[3])), (241, 196, 15), 1)
        for tb in self.tables:
            col = COLORS[tb.status]
            x1, y1, x2, y2 = map(int, tb.box)
            cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
            dur = fmt_dur(t - tb.session_start) if tb.session_start else '00:00:00'
            txt = f"T{tb.id} {tb.status} [{dur}] ({tb.persons}p)"
            cv2.putText(out, txt, (x1, max(20, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 2)
        return out
'''
with open("/content/cafe_project/cafe_monitor_v2.py", "w") as f:
    f.write(monitor_code)

# Exécution sur la partie de test de la vidéo
from cafe_monitor_v2 import CafeMonitorV2
mon = CafeMonitorV2(best_weights, conf=0.35)

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(total_frames * 0.75))
out_video_path = "/content/cafe_project/resultat_test.mp4"
writer = cv2.VideoWriter(out_video_path, cv2.VideoWriter_fourcc(*'mp4v'), 10, (w_vid, h_vid))

print("Génération de la vidéo de test (300 frames)... affichage avec chronomètres.")
for f_i in range(300):
    ret, fr = cap.read()
    if not ret:
        break
    sim_time = f_i * 1.5  # Simulation accélérée
    ann = mon.process(fr, t=sim_time)
    writer.write(ann)
cap.release()
writer.release()
print(f"✅ Vidéo de résultat générée : {out_video_path}")

## Étape 10 : Téléchargement des Modèles pour le Raspberry Pi
Téléchargez les fichiers générés pour les placer dans votre Raspberry Pi :

In [ ]:
from google.colab import files

print("Téléchargement du modèle PyTorch best.pt...")
files.download(best_weights)

if os.path.exists(onnx_file):
    print("Téléchargement du modèle ONNX pour Raspberry Pi...")
    files.download(onnx_file)

print("🎉 Félicitations ! Votre modèle est prêt pour la Raspberry Pi !")